# CNN HIV 5-Fold Statistics Revize
This notebook rebuilds the CNN model from the PDF with 5-fold stratified cross-validation and produces mean ± std and variance for Accuracy, Precision, Recall, F1, and ROC-AUC.

In [ ]:
!pip install rdkit
!pip install tensorflow

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
import random as rn
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow import keras
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from tensorflow.keras.utils import Sequence

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
%matplotlib inline

%tensorflow_version 2.x
import tensorflow as tf

In [ ]:

import os, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
N_SPLITS = 5
EPOCHS = 200
BATCH_SIZE = 128
PATIENCE = 10

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)
print(tf.__version__)


In [ ]:

# Load the dataset
df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")

# Target variable
y = df["HIV_active"].values

# Select numeric columns
X = df.drop(columns=["HIV_active"])
X = X.select_dtypes(include=["float64", "int64"]).values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class distribution:", pd.Series(y).value_counts().to_dict())


In [13]:

def build_cnn_model(input_dim):
    # Based on the simplified CNN architecture in the PDF
    model = models.Sequential([
        layers.Input(shape=(input_dim, 1)),
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.GlobalMaxPooling1D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.15),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model


In [14]:

def evaluate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

def plot_avg_training_history(all_histories, max_epoch=None):
    if not all_histories:
        return

    keys = ["loss", "val_loss", "accuracy", "val_accuracy", "auc", "val_auc"]
    avg_history = {}

    if max_epoch is None:
        max_epoch = max(len(h["loss"]) for h in all_histories)

    for key in keys:
        padded = []
        for h in all_histories:
            vals = h.get(key, [])
            if len(vals) < max_epoch:
                vals = vals + [vals[-1]] * (max_epoch - len(vals))
            else:
                vals = vals[:max_epoch]
            padded.append(vals)
        avg_history[key] = np.mean(np.array(padded), axis=0)

    plt.figure(figsize=(8,5))
    plt.plot(avg_history["accuracy"], label="Train Acc")
    plt.plot(avg_history["val_accuracy"], label="Val Acc")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.title("CNN Average Accuracy across 5 folds")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(avg_history["loss"], label="Train Loss")
    plt.plot(avg_history["val_loss"], label="Val Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("CNN Average Loss across 5 folds")
    plt.legend()
    plt.grid(True)
    plt.show()


In [15]:

def run_cnn_5fold(X, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_results = []
    all_histories = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        print("\n" + "="*30)
        print(f"FOLD {fold}/{n_splits}")
        print("="*30)

        set_seed(seed + fold)

        X_train_full, X_test = X[train_idx], X[test_idx]
        y_train_full, y_test = y[train_idx], y[test_idx]

        X_train, X_val, y_train, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.2,
            random_state=seed + fold,
            stratify=y_train_full
        )

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

        X_train_cnn = X_train[..., np.newaxis]
        X_val_cnn = X_val[..., np.newaxis]
        X_test_cnn = X_test[..., np.newaxis]

        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=np.unique(y_train),
            y=y_train
        )
        cw = dict(enumerate(class_weights))

        model = build_cnn_model(X_train.shape[1])

        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=PATIENCE,
            restore_best_weights=True
        )

        history = model.fit(
            X_train_cnn, y_train,
            validation_data=(X_val_cnn, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop],
            class_weight=cw,
            verbose=0
        )

        all_histories.append(history.history)

        y_prob = model.predict(X_test_cnn, verbose=0).ravel()
        metrics = evaluate_metrics(y_test, y_prob, threshold=0.5)
        metrics["fold"] = fold
        metrics["epochs_ran"] = len(history.history["loss"])
        fold_results.append(metrics)

        print({
            "accuracy": round(metrics["accuracy"], 4),
            "precision": round(metrics["precision"], 4),
            "recall": round(metrics["recall"], 4),
            "f1": round(metrics["f1"], 4),
            "roc_auc": round(metrics["roc_auc"], 4),
            "epochs_ran": metrics["epochs_ran"]
        })

    results_df = pd.DataFrame(fold_results)

    summary = {}
    metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    for col in metric_cols:
        summary[col] = {
            "mean": results_df[col].mean(),
            "std": results_df[col].std(ddof=1),
            "var": results_df[col].var(ddof=1),
            "formatted": f"{results_df[col].mean():.4f} ± {results_df[col].std(ddof=1):.4f}"
        }

    return results_df, summary, all_histories


In [ ]:

results_df, summary, all_histories = run_cnn_5fold(X, y, n_splits=N_SPLITS, seed=SEED)

print("\nFold-wise results:")
display(results_df)

summary_rows = []
for metric_name, vals in summary.items():
    summary_rows.append({
        "Metric": metric_name.upper(),
        "Mean": vals["mean"],
        "Std": vals["std"],
        "Variance": vals["var"],
        "Formatted": vals["formatted"]
    })

summary_df = pd.DataFrame(summary_rows)
print("\nSummary results:")
display(summary_df)

results_df.to_csv("CNN_fold_results.csv", index=False)
summary_df.to_csv("CNN_summary_results.csv", index=False)

print("\nSaved files:")
print("- CNN_fold_results.csv")
print("- CNN_summary_results.csv")


In [ ]:

plot_avg_training_history(all_histories)


## CNN docking preparation block

This block is prepared for Colab.

Added structure:
- RDKit installation cell
- rebuild and retrain the best fold
- extract the top 10 candidates
- create the shared column format
- select the final 2 candidates
- colored 2D molecule drawing
- `.smi` docking file

In [ ]:
# Colab RDKit install
import sys, subprocess, pkgutil

if pkgutil.find_loader("rdkit") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit-pypi"])

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

print("RDKit OK")

In [ ]:
# ================================
# CNN FINAL PIPELINE (ROBUST VERSION)
# ================================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping

# 0) automatically find the AUC column
candidate_auc_cols = [
    "test_roc_auc",
    "roc_auc",
    "val_roc_auc",
    "auc",
    "val_auc",
    "test_auc"
]

available_auc_cols = [c for c in candidate_auc_cols if c in results_df.columns]

if len(available_auc_cols) > 0:
    auc_col = available_auc_cols[0]
    best_fold = int(results_df[auc_col].astype(float).idxmax())
    print(f"Using AUC column: {auc_col}")
else:
    best_fold = len(results_df) - 1
    print("No ROC-AUC column found in results_df. Using last fold.")

best_fold_number = best_fold + 1
print(f"Using best fold: {best_fold_number}")

# 1) check smiles column
if "smiles" not in df.columns:
    raise ValueError("The dataset does not contain a 'smiles' column.")

# 2) Rebuild the same fold
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(skf.split(X, y))
train_idx, test_idx = splits[best_fold]

set_seed(SEED + best_fold_number)

X_train_full, X_test = X[train_idx], X[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
smiles_test = df.iloc[test_idx]["smiles"].reset_index(drop=True)

# 3) Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=SEED + best_fold_number,
    stratify=y_train_full
)

# 4) Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_cnn = X_train[..., np.newaxis]
X_val_cnn = X_val[..., np.newaxis]
X_test_cnn = X_test_scaled[..., np.newaxis]

# 5) Class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
cw = dict(enumerate(class_weights))

# 6) Rebuild and train the model
model = build_cnn_model(X_train.shape[1])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True
)

history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    class_weight=cw,
    verbose=0
)

# 7) Generate predictions
y_prob = model.predict(X_test_cnn, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

df_pred = pd.DataFrame({
    "fold": best_fold_number,
    "smiles": smiles_test,
    "y_true": np.array(y_test),
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 8) Top 10 candidates
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

# 9) Descriptor hesaplama
def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = RdLipinski.NumHDonors(mol)
    hba = RdLipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = RdLipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

desc_rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        desc_rows.append(d)

df_desc = pd.DataFrame(desc_rows)

# 10) Shared column format
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nCNN TOP 10 CANDIDATES:")
display(df_desc)

# 11) Final 2 candidates
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nCNN FINAL 2 CANDIDATES:")
display(final_df)

# 12) Save
df_desc.to_csv("CNN_top_10_candidates.csv", index=False)
final_df.to_csv("CNN_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("CNN_docking_input.smi", index=False, header=False)

print("\nSaved: CNN_top_10_candidates.csv")
print("Saved: CNN_final_2_candidates.csv")
print("Saved: CNN_docking_input.smi")

# 13) Colored drawing
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(340, 340),
    legends=[
        f"CNN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"CNN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)